In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [3]:
# ============================================================
# SECTION — Interactive spectral slope vs solar elongation
# Single-year OR all-years view
# ============================================================

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.signal import detrend, welch
from ipywidgets import interact, Dropdown

# ------------------------------------------------------------
# REPO SETUP
# ------------------------------------------------------------
PROJECT_ROOT = Path.cwd().resolve().parent
DATA_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.io_utils import (
    load_dsn_data,
    load_horizons_daily_sep,
)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_ROOT:", DATA_ROOT)

# ------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------
YEARS = [2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014]

F_LOW = 3e-4
F_HIGH = 5e-2

DOPPLER_LIMIT_HZ = 5.0
MIN_SAMPLES_PER_DAY = 300

# ------------------------------------------------------------
# DAILY SPECTRAL SLOPE
# ------------------------------------------------------------
def compute_daily_slope(group):

    if len(group) < MIN_SAMPLES_PER_DAY:
        return np.nan

    t_sec = (
        group.index - group.index[0]
    ).total_seconds().to_numpy()

    if len(t_sec) < 3:
        return np.nan

    dt = np.nanmedian(np.diff(t_sec))

    if not np.isfinite(dt) or dt <= 0:
        return np.nan

    fs = 1.0 / dt

    doppler = group["doppler"].to_numpy(dtype=float)
    doppler = doppler[np.isfinite(doppler)]

    if len(doppler) < MIN_SAMPLES_PER_DAY:
        return np.nan

    # detrend Doppler
    doppler_dt = detrend(doppler, type="linear")

    # Doppler -> phase
    phase = 2 * np.pi * np.cumsum(doppler_dt) * dt
    phase_dt = detrend(phase, type="linear")

    nperseg = min(
        4096,
        max(256, len(phase_dt) // 4)
    )

    freqs, psd = welch(
        phase_dt,
        fs=fs,
        window="hann",
        nperseg=nperseg,
        noverlap=nperseg // 2,
        detrend=False,
        scaling="density",
    )

    mask = (
        (freqs >= F_LOW)
        & (freqs <= F_HIGH)
        & np.isfinite(psd)
        & (psd > 0)
    )

    if mask.sum() < 20:
        return np.nan

    slope = np.polyfit(
        np.log10(freqs[mask]),
        np.log10(psd[mask]),
        1,
    )[0]

    return slope


# ------------------------------------------------------------
# BUILD ALL YEARS ONCE
# ------------------------------------------------------------
all_years_data = []

for year in YEARS:

    print(f"\n========== PROCESSING {year} ==========")

    doppler_file = (
        DATA_ROOT / "dataByYear" / f"data_{year}.txt"
    )

    horizons_file = (
        DATA_ROOT / "Scint_analysis" / f"vex_{year}.txt"
    )

    if not doppler_file.exists():
        print("Missing:", doppler_file)
        continue

    if not horizons_file.exists():
        print("Missing:", horizons_file)
        continue

    # load DSN
    df = load_dsn_data(doppler_file)

    df["UTC_time"] = pd.to_datetime(
        df["UTC_time"],
        errors="coerce"
    )

    df = (
        df.dropna(subset=["UTC_time"])
          .sort_values("UTC_time")
          .set_index("UTC_time")
    )

    df = df[
        ~df.index.duplicated(keep="first")
    ]

    # quality filter
    df = df[
        np.abs(df["doppler"]) < DOPPLER_LIMIT_HZ
    ].copy()

    print("DSN samples:", len(df))

    daily_rows = []

    # compute daily slopes
    for day, group in df.groupby(df.index.floor("D")):

        slope = compute_daily_slope(group)

        if np.isfinite(slope):

            daily_rows.append({
                "day": day,
                "year": year,
                "slope": slope,
                "n_samples": len(group),
            })

    daily_df = pd.DataFrame(daily_rows)

    if daily_df.empty:
        print("No valid daily rows.")
        continue

    # load SEP
    horizons_daily = load_horizons_daily_sep(
        horizons_file
    )

    merged = daily_df.merge(
        horizons_daily[["day", "elongation_deg"]],
        on="day",
        how="inner",
    )

    merged = merged.sort_values(
        "elongation_deg"
    )

    print("Merged rows:", len(merged))

    if merged.empty:
        continue

    all_years_data.append(merged)

# combined dataframe
combined_df = pd.concat(
    all_years_data,
    ignore_index=True
)

print("\nTOTAL ROWS:", len(combined_df))


# ------------------------------------------------------------
# INTERACTIVE PLOTTING
# ------------------------------------------------------------
def plot_spectral_slope(selected_year):

    fig, ax = plt.subplots(figsize=(8, 6))

    # --------------------------------------------------------
    # ALL YEARS
    # --------------------------------------------------------
    if selected_year == "All":

        for year in YEARS:

            sub = combined_df[
                combined_df["year"] == year
            ]

            if len(sub) == 0:
                continue

            ax.scatter(
                sub["elongation_deg"],
                sub["slope"],
                s=25,
                alpha=0.7,
                label=str(year),
            )

        ax.legend(
            loc="upper right",
            fontsize=9,
            frameon=True,
        )

        ax.set_title(
            "Spectral slope vs solar elongation\n(All years)"
        )

    # --------------------------------------------------------
    # SINGLE YEAR
    # --------------------------------------------------------
    else:

        sub = combined_df[
            combined_df["year"] == selected_year
        ]

        ax.scatter(
            sub["elongation_deg"],
            sub["slope"],
            s=30,
            alpha=0.8,
            label=str(selected_year),
        )

        # optional trend line
        if len(sub) > 3:

            coef = np.polyfit(
                sub["elongation_deg"],
                sub["slope"],
                1
            )

            xfit = np.linspace(
                sub["elongation_deg"].min(),
                sub["elongation_deg"].max(),
                200,
            )

            yfit = np.polyval(coef, xfit)

            ax.plot(
                xfit,
                yfit,
                linewidth=2,
                label="Linear trend",
            )

            corr = np.corrcoef(
                sub["elongation_deg"],
                sub["slope"]
            )[0, 1]

            print(
                f"\nYEAR {selected_year}"
            )

            print(
                "Correlation:",
                round(corr, 3)
            )

            print(
                "Mean slope:",
                round(sub["slope"].mean(), 3)
            )

        ax.legend(
            loc="upper right",
            fontsize=9,
            frameon=True,
        )

        ax.set_title(
            f"Spectral slope vs solar elongation ({selected_year})"
        )

    # --------------------------------------------------------
    # AXES
    # --------------------------------------------------------
    ax.set_xlabel(
        "Solar elongation, SEP (deg)"
    )

    ax.set_ylabel(
        "Phase PSD spectral slope"
    )

    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


# ------------------------------------------------------------
# WIDGET
# ------------------------------------------------------------
interact(
    plot_spectral_slope,
    selected_year=Dropdown(
        options=["All"] + YEARS,
        value="All",
        description="Year:",
    ),
)

print("\nREADY")

PROJECT_ROOT: /mnt/data/jhub/16-VenusExpres/dsn_multi_year
DATA_ROOT: /mnt/data/jhub/16-VenusExpres

========== PROCESSING 2006 ==========
DSN samples: 471602
Merged rows: 185

========== PROCESSING 2007 ==========
DSN samples: 1000771
Merged rows: 329

========== PROCESSING 2008 ==========
DSN samples: 855555
Merged rows: 338

========== PROCESSING 2009 ==========
DSN samples: 865652
Merged rows: 297

========== PROCESSING 2010 ==========
DSN samples: 867761
Merged rows: 334

========== PROCESSING 2011 ==========
DSN samples: 803281
Merged rows: 324

========== PROCESSING 2012 ==========
DSN samples: 927028
Merged rows: 322

========== PROCESSING 2013 ==========
DSN samples: 789453
Merged rows: 334

========== PROCESSING 2014 ==========
DSN samples: 264517
Merged rows: 120

TOTAL ROWS: 2583


interactive(children=(Dropdown(description='Year:', options=('All', 2006, 2007, 2008, 2009, 2010, 2011, 2012, …


READY
